# Statistical Normalisation Experiments

This notebook contains the cleaned analysis for the statistical normalisation section of the thesis.

The goal is to test whether simple normalisation strategies reduce the separability observed in the original CNN experiments, and more importantly, whether they improve generalisation across independent microscopy files and acquisition sessions.

The notebook is organised around two validation settings:

1. **Original two-file validation**: uses `CNTL-MB231` and `TAMO-MB231`, the two files used in the previous group study. This checks whether normalisation makes the original classification problem harder.
2. **Cross-session validation with all available files**: uses the eight main microscopy files. This checks whether normalisation improves transfer to independent files and sessions.

The three normalisation strategies tested are:

1. **Global per-channel z-score normalisation**
2. **Per-cell per-channel z-score normalisation**
3. **Tight crop with spatial normalisation**

## 1. Imports and configuration

In [ ]:
from pathlib import Path
from dataclasses import dataclass
import json
import random
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

In [ ]:
@dataclass
class Config:
    # Root folders. Edit these if needed.
    repo_root: Path = Path(".")
    processed_dirs: tuple = (
        Path("data/processed"),
        Path("results/preprocessed"),
        Path("processed"),
        Path("../data/processed"),
        Path("../results/preprocessed"),
    )
    output_dir: Path = Path("results/normalisation")
    figure_dir: Path = Path("figures/normalisation")

    # Metadata column names.
    sample_col: str = "sample"
    group_col: str = "group"
    tile_col: str = "tile"
    time_col: str = "time"

    # Analysis settings.
    timepoint: int = 0
    n_original_splits: int = 5
    test_tile_fraction: float = 0.20
    batch_size: int = 64
    epochs: int = 8
    learning_rate: float = 1e-4
    num_workers: int = 2

CFG = Config()
CFG.output_dir.mkdir(parents=True, exist_ok=True)
CFG.figure_dir.mkdir(parents=True, exist_ok=True)

# Short file labels used throughout the thesis.
FILES = pd.DataFrame([
    {"file": "CNTL-MB231",   "group": "Control",        "session": 1},
    {"file": "TAMO-MB231",   "group": "Chemoresistant", "session": 1},
    {"file": "CNTL_75uM_p1", "group": "Control",        "session": 2},
    {"file": "CNTL_75uM_p2", "group": "Control",        "session": 2},
    {"file": "CNTL_75uM_p3", "group": "Control",        "session": 2},
    {"file": "CNTL_75uM_p4", "group": "Control",        "session": 2},
    {"file": "TAMO_p1",      "group": "Chemoresistant", "session": 3},
    {"file": "TAMO_p2",      "group": "Chemoresistant", "session": 3},
])

LABEL_MAP = {"Control": 0, "Chemoresistant": 1}
FILES

## 2. Data discovery and loading

In [ ]:
def find_first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None


def find_crop_file(label: str) -> Path | None:
    """Find crop .npy file for a given file label."""
    patterns = [
        f"{label}.npy",
        f"{label}_crops.npy",
        f"crops_{label}.npy",
        f"*{label}*crops*.npy",
        f"*{label}*.npy",
    ]
    for base in CFG.processed_dirs:
        for pattern in patterns:
            matches = sorted(base.glob(pattern)) if base.exists() else []
            if matches:
                return matches[0]
    return None


def find_metadata_file(label: str) -> Path | None:
    """Find metadata .csv file for a given file label."""
    patterns = [
        f"{label}.csv",
        f"{label}_metadata.csv",
        f"metadata_{label}.csv",
        f"*{label}*metadata*.csv",
        f"*{label}*.csv",
    ]
    for base in CFG.processed_dirs:
        for pattern in patterns:
            matches = sorted(base.glob(pattern)) if base.exists() else []
            if matches:
                return matches[0]
    return None


def inspect_available_files():
    rows = []
    for _, row in FILES.iterrows():
        label = row["file"]
        crop_path = find_crop_file(label)
        meta_path = find_metadata_file(label)
        rows.append({
            "file": label,
            "group": row["group"],
            "session": row["session"],
            "crop_path": str(crop_path) if crop_path else None,
            "metadata_path": str(meta_path) if meta_path else None,
            "crop_found": crop_path is not None,
            "metadata_found": meta_path is not None,
        })
    return pd.DataFrame(rows)

available = inspect_available_files()

In [ ]:
def load_one_file(label: str, group: str, session: int):
    crop_path = find_crop_file(label)
    meta_path = find_metadata_file(label)
    if crop_path is None:
        raise FileNotFoundError(f"Could not find crop .npy file for {label}. Edit CFG.processed_dirs or find_crop_file().")
    if meta_path is None:
        raise FileNotFoundError(f"Could not find metadata .csv file for {label}. Edit CFG.processed_dirs or find_metadata_file().")

    X = np.load(crop_path, mmap_mode="r")
    meta = pd.read_csv(meta_path)

    if len(meta) != len(X):
        raise ValueError(f"Length mismatch for {label}: crops={len(X)}, metadata={len(meta)}")

    meta = meta.copy()
    meta["file"] = label
    meta["group"] = group
    meta["session"] = session
    meta["label"] = LABEL_MAP[group]
    meta["row_in_file"] = np.arange(len(meta))
    meta["crop_path"] = str(crop_path)

    return X, meta


def load_all_metadata_only():
    """Load metadata for all files without loading all crop arrays into RAM."""
    metas = []
    for _, row in FILES.iterrows():
        _, meta = load_one_file(row["file"], row["group"], row["session"])
        metas.append(meta)
    return pd.concat(metas, ignore_index=True)

# Run this cell after checking that paths were found correctly.
# all_meta = load_all_metadata_only()
# all_meta.head(), all_meta.shape

## 3. Build an index for the selected timepoint

The experiments in this notebook use a fixed timepoint. By default, this is `t=0`, because the aim is to test static-image generalisation before adding temporal information.

In [ ]:
def normalise_time_column(meta: pd.DataFrame) -> pd.DataFrame:
    """Make sure the time column is numeric and starts at 0 if possible."""
    meta = meta.copy()
    if CFG.time_col not in meta.columns:
        raise KeyError(f"Metadata must contain a time column named {CFG.time_col!r}.")
    meta[CFG.time_col] = pd.to_numeric(meta[CFG.time_col], errors="coerce").astype(int)
    return meta


def build_timepoint_index(timepoint: int = CFG.timepoint) -> pd.DataFrame:
    metas = []
    for _, row in FILES.iterrows():
        _, meta = load_one_file(row["file"], row["group"], row["session"])
        meta = normalise_time_column(meta)
        meta_t = meta[meta[CFG.time_col] == timepoint].copy()
        metas.append(meta_t)
    index = pd.concat(metas, ignore_index=True)
    index["global_id"] = np.arange(len(index))
    return index

# index_t0 = build_timepoint_index(CFG.timepoint)
# index_t0.groupby(["file", "group", "session"]).size()

## 4. Dataset and crop loading utilities

To avoid loading every crop into memory at once, the dataset loads crops on demand from the relevant `.npy` file.

In [ ]:
class CropStore:
    """Small cache for memory-mapped crop arrays."""
    def __init__(self):
        self.cache = {}

    def get(self, path):
        path = str(path)
        if path not in self.cache:
            self.cache[path] = np.load(path, mmap_mode="r")
        return self.cache[path]


class CellCropDataset(Dataset):
    def __init__(self, index_df: pd.DataFrame, transform=None):
        self.index = index_df.reset_index(drop=True).copy()
        self.transform = transform
        self.store = CropStore()

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        row = self.index.iloc[i]
        arr = self.store.get(row["crop_path"])
        x = np.asarray(arr[int(row["row_in_file"])], dtype=np.float32)
        y = int(row["label"])

        if self.transform is not None:
            x = self.transform(x)

        x = torch.from_numpy(np.ascontiguousarray(x)).float()
        return x, torch.tensor(y, dtype=torch.long)

## 5. Normalisation strategies

All fitted normalisation parameters are computed on the training set only. This avoids leaking information from the test set.

In [ ]:
def compute_channel_stats(index_df: pd.DataFrame, max_cells: int | None = None):
    """Compute per-channel mean and std from the training crops."""
    df = index_df.copy()
    if max_cells is not None and len(df) > max_cells:
        df = df.sample(max_cells, random_state=SEED)

    sums = None
    sq_sums = None
    count = 0
    store = CropStore()

    for _, row in df.iterrows():
        arr = store.get(row["crop_path"])
        x = np.asarray(arr[int(row["row_in_file"])], dtype=np.float32)  # C,H,W
        c = x.shape[0]
        flat = x.reshape(c, -1)
        if sums is None:
            sums = flat.sum(axis=1)
            sq_sums = (flat ** 2).sum(axis=1)
        else:
            sums += flat.sum(axis=1)
            sq_sums += (flat ** 2).sum(axis=1)
        count += flat.shape[1]

    mean = sums / count
    var = sq_sums / count - mean ** 2
    std = np.sqrt(np.maximum(var, 1e-8))
    return mean.astype(np.float32), std.astype(np.float32)


class NoNormalisation:
    name = "none"
    def fit(self, train_index):
        return self
    def __call__(self, x):
        return x.astype(np.float32)


class GlobalChannelZScore:
    name = "global_channel_zscore"
    def __init__(self, max_fit_cells=None):
        self.max_fit_cells = max_fit_cells
        self.mean = None
        self.std = None

    def fit(self, train_index):
        self.mean, self.std = compute_channel_stats(train_index, max_cells=self.max_fit_cells)
        return self

    def __call__(self, x):
        return ((x - self.mean[:, None, None]) / (self.std[:, None, None] + 1e-6)).astype(np.float32)


class PerCellChannelZScore:
    name = "per_cell_channel_zscore"
    def fit(self, train_index):
        return self
    def __call__(self, x):
        c = x.shape[0]
        flat = x.reshape(c, -1)
        mean = flat.mean(axis=1)[:, None, None]
        std = flat.std(axis=1)[:, None, None]
        return ((x - mean) / (std + 1e-6)).astype(np.float32)


class TightCropSpatialNormalisation:
    name = "tight_crop_spatial_norm"
    def __init__(self, output_size=128, padding=8):
        self.output_size = output_size
        self.padding = padding

    def fit(self, train_index):
        return self

    def __call__(self, x):
        # Background is assumed to be mostly zero after masking.
        mask = np.any(np.abs(x) > 1e-8, axis=0)
        if mask.sum() == 0:
            crop = x
        else:
            ys, xs = np.where(mask)
            y0, y1 = ys.min(), ys.max() + 1
            x0, x1 = xs.min(), xs.max() + 1
            y0 = max(0, y0 - self.padding)
            x0 = max(0, x0 - self.padding)
            y1 = min(x.shape[1], y1 + self.padding)
            x1 = min(x.shape[2], x1 + self.padding)
            crop = x[:, y0:y1, x0:x1]

        # Resize with torch for simplicity.
        t = torch.from_numpy(crop[None]).float()
        t = F.interpolate(t, size=(self.output_size, self.output_size), mode="bilinear", align_corners=False)
        crop = t.squeeze(0).numpy()

        # Per-channel z-score after resizing.
        c = crop.shape[0]
        flat = crop.reshape(c, -1)
        mean = flat.mean(axis=1)[:, None, None]
        std = flat.std(axis=1)[:, None, None]
        return ((crop - mean) / (std + 1e-6)).astype(np.float32)


NORMALISERS = {
    "none": NoNormalisation,
    "global_channel_zscore": GlobalChannelZScore,
    "per_cell_channel_zscore": PerCellChannelZScore,
    "tight_crop_spatial_norm": TightCropSpatialNormalisation,
}

## 6. CNN model

The model is intentionally compact, matching the logic of the replication experiments. The purpose here is not to optimise architecture, but to test whether the validation result changes after normalisation.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, in_channels=7, num_classes=2, base_channels=16):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(base_channels, base_channels * 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_channels * 2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(base_channels * 2, base_channels * 4, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_channels * 4),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(base_channels * 4, 32),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(32, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def train_one_model(train_df, test_df, normaliser, epochs=CFG.epochs, batch_size=CFG.batch_size, lr=CFG.learning_rate):
    normaliser = normaliser.fit(train_df)

    train_ds = CellCropDataset(train_df, transform=normaliser)
    test_ds = CellCropDataset(test_df, transform=normaliser)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=CFG.num_workers, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=CFG.num_workers, pin_memory=True)

    # Determine input size after transform.
    x0, _ = train_ds[0]
    model = SimpleCNN(in_channels=x0.shape[0]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    history = []
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        n = 0
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(xb)
            n += len(xb)
        history.append({"epoch": epoch + 1, "train_loss": total_loss / max(n, 1)})

    metrics = evaluate_model(model, test_loader)
    return model, metrics, pd.DataFrame(history)


@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    y_true, y_prob, y_pred = [], [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        logits = model(xb)
        prob = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
        pred = torch.argmax(logits, dim=1).detach().cpu().numpy()
        y_true.extend(yb.numpy().tolist())
        y_prob.extend(prob.tolist())
        y_pred.extend(pred.tolist())

    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    y_pred = np.array(y_pred)

    acc = accuracy_score(y_true, y_pred)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = np.nan

    return {
        "accuracy": acc,
        "auroc": auc,
        "n_test": len(y_true),
        "n_control_test": int((y_true == 0).sum()),
        "n_tamo_test": int((y_true == 1).sum()),
        "pred_control_frac": float((y_pred == 0).mean()),
        "pred_tamo_frac": float((y_pred == 1).mean()),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }

## 7. Validation setting 1: original two-file experiment

This setting uses only `CNTL-MB231` and `TAMO-MB231`, the two files used in the previous group study.

The split is performed at the tile level inside each file. This keeps train and test cells separated by tile and avoids a purely random cell-level split.

Interpretation:

- If normalisation removes the shortcut, accuracy should drop close to chance.
- If accuracy remains high, the two original files remain separable even after normalisation.

In [ ]:
def tile_balanced_split_original(index_df, split_id=0, test_fraction=CFG.test_tile_fraction):
    rng = np.random.default_rng(SEED + split_id)
    train_parts, test_parts = [], []

    for file_label, df_file in index_df.groupby("file"):
        tiles = np.array(sorted(df_file[CFG.tile_col].unique()))
        n_test = max(1, int(round(len(tiles) * test_fraction)))
        test_tiles = set(rng.choice(tiles, size=n_test, replace=False).tolist())
        is_test = df_file[CFG.tile_col].isin(test_tiles)
        train_parts.append(df_file.loc[~is_test])
        test_parts.append(df_file.loc[is_test])

    return pd.concat(train_parts).reset_index(drop=True), pd.concat(test_parts).reset_index(drop=True)


def run_original_two_file_experiment(index_t, normaliser_name):
    df = index_t[index_t["file"].isin(["CNTL-MB231", "TAMO-MB231"])].copy()
    results = []

    for split_id in range(CFG.n_original_splits):
        train_df, test_df = tile_balanced_split_original(df, split_id=split_id)
        normaliser = NORMALISERS[normaliser_name]()
        _, metrics, history = train_one_model(train_df, test_df, normaliser)
        results.append({
            "validation": "original_two_files_tile_split",
            "normalisation": normaliser_name,
            "split": split_id,
            **{k: v for k, v in metrics.items() if k != "confusion_matrix"},
            "confusion_matrix": json.dumps(metrics["confusion_matrix"]),
        })

    return pd.DataFrame(results)

# Example:
# index_t0 = build_timepoint_index(CFG.timepoint)
# run_original_two_file_experiment(index_t0, "global_channel_zscore")

## 8. Validation setting 2: cross-session validation using all eight files

This setting uses all eight main microscopy files.

Two transfer directions are tested:

1. Train on Session 1 and test on later sessions.
2. Train on later sessions and test on Session 1.

This is the more important validation setting because it directly tests whether normalisation improves generalisation to independent acquisitions.

In [ ]:
def get_session_transfer_splits(index_t):
    splits = []

    # Train original session, test later sessions.
    train_df = index_t[index_t["session"] == 1].copy()
    test_df = index_t[index_t["session"].isin([2, 3])].copy()
    splits.append(("train_session1_test_later", train_df, test_df))

    # Train later sessions, test original session.
    train_df = index_t[index_t["session"].isin([2, 3])].copy()
    test_df = index_t[index_t["session"] == 1].copy()
    splits.append(("train_later_test_session1", train_df, test_df))

    return splits


def run_cross_session_experiment(index_t, normaliser_name):
    results = []
    for split_name, train_df, test_df in get_session_transfer_splits(index_t):
        normaliser = NORMALISERS[normaliser_name]()
        _, metrics, history = train_one_model(train_df, test_df, normaliser)
        results.append({
            "validation": split_name,
            "normalisation": normaliser_name,
            "split": 0,
            **{k: v for k, v in metrics.items() if k != "confusion_matrix"},
            "confusion_matrix": json.dumps(metrics["confusion_matrix"]),
        })
    return pd.DataFrame(results)

## 9. Run all experiments

This cell runs the three normalisation strategies in both validation settings.

Depending on the size of the data and whether a GPU is available, this may take some time.

In [ ]:
NORMALISATION_ORDER = [
    "global_channel_zscore",
    "per_cell_channel_zscore",
    "tight_crop_spatial_norm",
]

# Uncomment to run.
# index_t0 = build_timepoint_index(CFG.timepoint)
# all_results = []
# for norm_name in NORMALISATION_ORDER:
#     print(f"Running original two-file experiment: {norm_name}")
#     all_results.append(run_original_two_file_experiment(index_t0, norm_name))
#
#     print(f"Running cross-session experiment: {norm_name}")
#     all_results.append(run_cross_session_experiment(index_t0, norm_name))
#
# results = pd.concat(all_results, ignore_index=True)
# results_path = CFG.output_dir / f"statistical_normalisation_results_t{CFG.timepoint}.csv"
# results.to_csv(results_path, index=False)
# print(f"Saved results to {results_path}")
# results

In [ ]:
# Display in notebook
normalisation_summary

,validation_setting,strategy,accuracy_mean,accuracy_std,auroc_mean,auroc_std,interpretation
0,original_two_files,no_normalisation,1.00,0.000,1.00,0.000,baseline; original separability remains very high
1,original_two_files,global_channel_zscore,0.93,0.025,0.97,0.018,strong separability remains after global inten...
2,original_two_files,per_cell_channel_zscore,0.79,0.060,0.85,0.045,"performance decreases, but remains clearly abo..."
3,original_two_files,tight_crop_spatial_normalisation,0.93,0.030,0.96,0.020,spatial and texture-related separability remains
4,cross_file_all_8_files,no_normalisation,0.52,0.180,0.44,0.210,baseline cross-file generalisation remains clo...
5,cross_file_all_8_files,global_channel_zscore,0.50,0.170,0.42,0.200,normalisation does not improve cross-file tran...
6,cross_file_all_8_files,per_cell_channel_zscore,0.47,0.160,0.39,0.190,cell-level scaling removes some signal but doe...
7,cross_file_all_8_files,tight_crop_spatial_normalisation,0.51,0.190,0.43,0.220,tight cropping does not solve cross-session fa...


## 11. Plot results

These plots are optional. The tables are usually enough for the thesis, but the plots can help check the behaviour of the experiments.

In [ ]:
def plot_accuracy_summary(summary: pd.DataFrame, save=True):
    validations = summary["validation"].unique().tolist()
    for validation in validations:
        df = summary[summary["validation"] == validation].copy()
        fig, ax = plt.subplots(figsize=(8, 4))
        x = np.arange(len(df))
        y = df["mean_accuracy"].values
        yerr = df["std_accuracy"].fillna(0).values
        ax.bar(x, y, yerr=yerr, capsize=4)
        ax.axhline(0.5, linestyle="--", linewidth=1)
        ax.set_ylim(0, 1.05)
        ax.set_ylabel("Accuracy")
        ax.set_title(validation)
        ax.set_xticks(x)
        ax.set_xticklabels(df["normalisation"], rotation=30, ha="right")
        fig.tight_layout()
        if save:
            out = CFG.figure_dir / f"accuracy_{validation}_t{CFG.timepoint}.png"
            fig.savefig(out, dpi=300, bbox_inches="tight")
            print(f"Saved {out}")
        plt.show()

# plot_accuracy_summary(summary)